In [0]:
# ==========================================================
# 01_gold_indicador_municipio
# Camada: Gold
# Fontes: default.indicador_municipio (Silver)
#         default.meta_municipio (Silver)
# Destino: default.gold_indicador_municipio
#
# RESPONSABILIDADE
# Consolidar indicador observado e meta oficial por município
# e ano, calculando gap e status de atingimento.
#
# DECISÕES DE MODELAGEM
# 1. Granularidade: município x ano.
# 2. Casos sem dado são MANTIDOS e classificados.
# 3. A meta é pivotada para meta_do_ano.
#
# NOTA SOBRE O JOIN
# O join é por (ano, id_municipio). A coluna rede NÃO entra
# na chave porque na Silver de meta ela é texto ('Municipal')
# e no indicador é código numérico.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA = "default"

TABELA_INDICADOR = f"{CATALOGO}.{SCHEMA}.indicador_municipio"
TABELA_META      = f"{CATALOGO}.{SCHEMA}.meta_municipio"
TABELA_DESTINO   = f"{CATALOGO}.{SCHEMA}.gold_indicador_municipio"

print(f"Indicador : {TABELA_INDICADOR}")
print(f"Meta      : {TABELA_META}")
print(f"Destino   : {TABELA_DESTINO}")

In [0]:
# ==========================================================
# Célula 2 — Leitura das Silver
# ==========================================================

df_ind = spark.table(TABELA_INDICADOR)
df_meta = spark.table(TABELA_META)

print(f"Indicador: {df_ind.count():,} registros | {len(df_ind.columns)} colunas")
print(f"Meta     : {df_meta.count():,} registros | {len(df_meta.columns)} colunas")

print("\nSchema indicador:")
df_ind.printSchema()

print("\nSchema meta:")
df_meta.printSchema()

In [0]:
# ==========================================================
# Célula 3 — Agregação do indicador por município
#
# PROBLEMA: o indicador tem uma linha por (ano, municipio,
# rede). A meta tem uma linha por (ano, municipio).
#
# SOLUÇÃO: agregar as redes por média simples da taxa,
# contando quantas redes contribuíram. A coluna qtd_redes
# documenta a cobertura e permite filtrar depois.
#
# LIMITAÇÃO CONHECIDA: sem contagem de alunos por rede, a
# média simples é uma aproximação. Está documentada na
# seção de limitações do documento de arquitetura.
# ==========================================================

df_ind_prep = (
    df_ind
    .groupBy("ano", "id_municipio")
    .agg(
        F.round(F.avg("taxa_alfabetizacao"), 4).alias("taxa_observada"),
        F.round(F.avg("media_portugues"), 4).alias("media_portugues"),
        F.countDistinct("rede").alias("qtd_redes"),
        F.count("*").alias("qtd_registros_rede"),
    )
)

print(f"Municípios-ano após agregação: {df_ind_prep.count():,}")
print("\nDistribuição de qtd_redes:")
df_ind_prep.groupBy("qtd_redes").count().orderBy("qtd_redes").show()

In [0]:
# ==========================================================
# Célula 4 — Preparação da meta
#
# A meta é pivotada: meta_do_ano recebe o valor da coluna
# meta_alfabetizacao_<ano> correspondente ao ano da linha.
#
# NOTA: 2023 não tem meta porque o compromisso do INEP começa
# em 2024. Essas linhas entram na Gold como linha de base,
# com status_meta = 'sem_meta'.
# ==========================================================

df_meta_prep = (
    df_meta
    .select(
        F.col("ano").cast("int").alias("ano"),
        F.col("id_municipio").cast("string").alias("id_municipio"),
        F.col("nivel_alfabetizacao").cast("int").alias("nivel_alfabetizacao"),
        F.col("percentual_participacao").cast("double").alias("percentual_participacao"),
        *[F.col(f"meta_alfabetizacao_{a}").cast("double").alias(f"meta_{a}") for a in range(2024, 2031)],
    )
    .withColumn(
        "meta_do_ano",
        F.when(F.col("ano") == 2024, F.col("meta_2024"))
         .when(F.col("ano") == 2025, F.col("meta_2025"))
         .when(F.col("ano") == 2026, F.col("meta_2026"))
         .when(F.col("ano") == 2027, F.col("meta_2027"))
         .when(F.col("ano") == 2028, F.col("meta_2028"))
         .when(F.col("ano") == 2029, F.col("meta_2029"))
         .when(F.col("ano") == 2030, F.col("meta_2030"))
         .otherwise(F.lit(None).cast("double"))
    )
    .dropDuplicates(["ano", "id_municipio"])
)

print(f"Meta preparada: {df_meta_prep.count():,}")

print("\nCobertura de meta_do_ano por ano:")
df_meta_prep.groupBy("ano").agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("meta_do_ano").isNotNull(), 1).otherwise(0)).alias("com_meta"),
    F.sum(F.when(F.col("meta_do_ano").isNull(), 1).otherwise(0)).alias("sem_meta"),
).orderBy("ano").show()

print("\nAmostra de 2024 (deve ter meta preenchida):")
df_meta_prep.filter(F.col("ano") == 2024) \
    .select("ano", "id_municipio", "meta_do_ano", "meta_2024", "meta_2025") \
    .show(5, truncate=False)

In [0]:
# ==========================================================
# Célula 5 — Join e cálculo do status
#
# Full outer join preserva municípios que existem em apenas
# uma das fontes.
#
# status_meta tem 4 valores:
#   atingiu   -> taxa >= meta
#   abaixo    -> taxa < meta
#   sem_meta  -> taxa existe, meta não
#   sem_dado  -> taxa não existe
# ==========================================================

df_gold = (
    df_ind_prep.alias("i")
    .join(df_meta_prep.alias("m"), on=["ano", "id_municipio"], how="full_outer")
    .select(
        F.coalesce(F.col("i.ano"), F.col("m.ano")).alias("ano"),
        F.coalesce(F.col("i.id_municipio"), F.col("m.id_municipio")).alias("id_municipio"),
        F.col("i.taxa_observada").alias("taxa_observada"),
        F.col("i.media_portugues").alias("media_portugues"),
        F.col("i.qtd_redes").alias("qtd_redes"),
        F.col("m.meta_do_ano").alias("meta_do_ano"),
        F.col("m.nivel_alfabetizacao").alias("nivel_alfabetizacao"),
        F.col("m.percentual_participacao").alias("percentual_participacao"),
        *[F.col(f"m.meta_{a}").alias(f"meta_{a}") for a in range(2024, 2031)],
    )
    .withColumn(
        "gap_pp",
        F.when(
            F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull(),
            F.round(F.col("taxa_observada") - F.col("meta_do_ano"), 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "status_meta",
        F.when(F.col("taxa_observada").isNull(), F.lit("sem_dado"))
         .when(F.col("meta_do_ano").isNull(), F.lit("sem_meta"))
         .when(F.col("taxa_observada") >= F.col("meta_do_ano"), F.lit("atingiu"))
         .otherwise(F.lit("abaixo"))
    )
    .withColumn(
        "tem_dado_completo",
        (F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull()).cast("boolean")
    )
)

print(f"Registros na Gold: {df_gold.count():,}")
print("\nDistribuição de status_meta:")
df_gold.groupBy("status_meta").count().orderBy(F.desc("count")).show()

In [0]:
# ==========================================================
# Célula 6 — Data Quality da Gold
# ==========================================================

total = df_gold.count()

nulos_chave = df_gold.filter(
    F.col("ano").isNull() | F.col("id_municipio").isNull()
).count()

chave = ["ano", "id_municipio"]
duplicatas = total - df_gold.dropDuplicates(chave).count()

status_invalido = df_gold.filter(
    ~F.col("status_meta").isin("atingiu", "abaixo", "sem_meta", "sem_dado")
).count()

gap_inconsistente = df_gold.filter(
    (F.col("gap_pp").isNotNull()) &
    (F.col("taxa_observada").isNull() | F.col("meta_do_ano").isNull())
).count()

print("=" * 55)
print("DATA QUALITY — gold_indicador_municipio")
print("=" * 55)
print(f"Registros totais       : {total:,}")
print(f"Chave nula             : {nulos_chave:,}")
print(f"Duplicatas na chave    : {duplicatas:,}")
print(f"Status fora do domínio : {status_invalido:,}")
print(f"Gap inconsistente      : {gap_inconsistente:,}")
print(f"Anos                   : {[r[0] for r in df_gold.select('ano').distinct().orderBy('ano').collect()]}")
print(f"Municípios distintos   : {df_gold.select('id_municipio').distinct().count():,}")
print("=" * 55)

assert nulos_chave == 0, "FALHA CRÍTICA: chave nula"
assert duplicatas == 0, "FALHA CRÍTICA: duplicatas na chave"
assert status_invalido == 0, "FALHA CRÍTICA: status_meta fora do domínio"
assert gap_inconsistente == 0, "FALHA CRÍTICA: gap calculado sem taxa ou meta"
print("Todas as regras críticas passaram.")

In [0]:
# ==========================================================
# Célula 7 — Gravação da Gold
# ==========================================================

(df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ano")
    .saveAsTable(TABELA_DESTINO))

print(f"Gravado em {TABELA_DESTINO}")
print(f"Registros: {spark.table(TABELA_DESTINO).count():,}")

In [0]:
# ==========================================================
# Célula 8 — Validação pós-gravação + ZORDER
# ==========================================================

df_check = spark.table(TABELA_DESTINO)

print(f"Registros na tabela : {df_check.count():,}")
print(f"Colunas             : {len(df_check.columns)}")
print(f"Anos                : {[r[0] for r in df_check.select('ano').distinct().orderBy('ano').collect()]}")

print("\nStatus por ano:")
df_check.groupBy("ano", "status_meta").count().orderBy("ano", "status_meta").show()

print("\nCobertura por ano:")
df_check.groupBy("ano").agg(
    F.count("*").alias("municipios"),
    F.sum(F.when(F.col("tem_dado_completo"), 1).otherwise(0)).alias("com_dado_completo"),
    F.sum(F.when(F.col("status_meta") == "atingiu", 1).otherwise(0)).alias("atingiu"),
    F.sum(F.when(F.col("status_meta") == "abaixo", 1).otherwise(0)).alias("abaixo"),
).orderBy("ano").show()

print("\nTop 10 municípios com maior gap positivo:")
df_check.filter(F.col("gap_pp").isNotNull()) \
    .orderBy(F.desc("gap_pp")) \
    .select("ano", "id_municipio", "taxa_observada", "meta_do_ano", "gap_pp", "qtd_redes") \
    .show(10, truncate=False)

print("\nTop 10 municípios com maior gap negativo:")
df_check.filter(F.col("gap_pp").isNotNull()) \
    .orderBy(F.asc("gap_pp")) \
    .select("ano", "id_municipio", "taxa_observada", "meta_do_ano", "gap_pp", "qtd_redes") \
    .show(10, truncate=False)

spark.sql(f"OPTIMIZE {TABELA_DESTINO} ZORDER BY (id_municipio)")
print("\nZORDER aplicado em id_municipio.")

In [0]:
# ==========================================================
# Diagnóstico de robustez dos extremos
# ==========================================================

print("Distribuição de taxa_observada em 2024:")
df_check.filter(F.col("ano") == 2024) \
    .select("taxa_observada") \
    .summary("count", "min", "25%", "50%", "75%", "max").show()

print("\nMunicípios com taxa = 100 em 2024:")
df_check.filter((F.col("ano") == 2024) & (F.col("taxa_observada") == 100.0)) \
    .select("id_municipio", "taxa_observada", "meta_do_ano", "gap_pp", "qtd_redes") \
    .show(20, truncate=False)

print("\nMunicípios com taxa < 20 em 2024:")
df_check.filter((F.col("ano") == 2024) & (F.col("taxa_observada") < 20)) \
    .select("id_municipio", "taxa_observada", "meta_do_ano", "gap_pp", "qtd_redes") \
    .show(20, truncate=False)

print("\nQuantos municípios têm taxa = 100 ou taxa < 20:")
df_check.filter(F.col("ano") == 2024).agg(
    F.sum(F.when(F.col("taxa_observada") == 100.0, 1).otherwise(0)).alias("taxa_100"),
    F.sum(F.when(F.col("taxa_observada") < 20, 1).otherwise(0)).alias("taxa_abaixo_20"),
    F.count("*").alias("total"),
).show()

In [0]:
# ==========================================================
# Célula 9 — Flag de outlier na Gold
#
# Critério: taxa = 100 ou taxa < 20 em 2024.
# Não exclui — apenas sinaliza para filtro posterior.
# ==========================================================

from delta.tables import DeltaTable

df_gold_flag = df_check.withColumn(
    "flag_outlier",
    F.when(
        (F.col("ano") == 2024) &
        ((F.col("taxa_observada") == 100.0) | (F.col("taxa_observada") < 20)),
        F.lit(True)
    ).otherwise(F.lit(False))
)

print("Distribuição de flag_outlier:")
df_gold_flag.groupBy("ano", "flag_outlier").count().orderBy("ano", "flag_outlier").show()

print("\nImpacto nos rankings de 2024:")
df_gold_flag.filter((F.col("ano") == 2024) & (F.col("gap_pp").isNotNull())).agg(
    F.count("*").alias("total_com_gap"),
    F.sum(F.when(F.col("flag_outlier"), 1).otherwise(0)).alias("outliers"),
    F.sum(F.when(~F.col("flag_outlier"), 1).otherwise(0)).alias("validos"),
).show()

# Regrava com a coluna nova
(df_gold_flag.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ano")
    .saveAsTable(TABELA_DESTINO))

print(f"\nRegravado com flag_outlier: {spark.table(TABELA_DESTINO).count():,} registros")

In [0]:
# ==========================================================
# Célula 10 — Rankings filtrados (sem outliers)
# ==========================================================

df_limpo = spark.table(TABELA_DESTINO).filter(
    (F.col("ano") == 2024) &
    (F.col("gap_pp").isNotNull()) &
    (~F.col("flag_outlier"))
)

print("Top 10 municípios com maior gap positivo (sem outliers):")
df_limpo.orderBy(F.desc("gap_pp")) \
    .select("id_municipio", "taxa_observada", "meta_do_ano", "gap_pp", "qtd_redes") \
    .show(10, truncate=False)

print("\nTop 10 municípios com maior gap negativo (sem outliers):")
df_limpo.orderBy(F.asc("gap_pp")) \
    .select("id_municipio", "taxa_observada", "meta_do_ano", "gap_pp", "qtd_redes") \
    .show(10, truncate=False)

print("\nEstatísticas de gap (sem outliers):")
df_limpo.select("gap_pp").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

In [0]:
# ==========================================================
# Diagnóstico de granularidade — UF
# ==========================================================

print("INDICADOR UF — serie:")
df_uf_ind.groupBy("serie").count().orderBy("serie").show()

print("\nINDICADOR UF — rede:")
df_uf_ind.groupBy("rede").count().orderBy("rede").show()

print("\nINDICADOR UF — pares (ano, sigla_uf) distintos:")
print(f"{df_uf_ind.select('ano','sigla_uf').distinct().count()}")

print("\nINDICADOR UF — pares (ano, sigla_uf, rede) distintos:")
print(f"{df_uf_ind.select('ano','sigla_uf','rede').distinct().count()}")

print("\nMETA UF — rede:")
df_uf_meta.groupBy("rede").count().orderBy("rede").show()

print("\nMETA UF — pares (ano, sigla_uf) distintos:")
print(f"{df_uf_meta.select('ano','sigla_uf').distinct().count()}")

print("\nMETA UF — duplicatas na chave (ano, sigla_uf):")
print(f"{df_uf_meta.count() - df_uf_meta.dropDuplicates(['ano','sigla_uf']).count()}")

print("\nUFs no indicador:")
print(sorted([r[0] for r in df_uf_ind.select("sigla_uf").distinct().collect()]))

print("\nUFs na meta:")
print(sorted([r[0] for r in df_uf_meta.select("sigla_uf").distinct().collect()]))